In [2]:
!pip install yfinance arch python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.3 MB/s eta 0:00:00


In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from arch import arch_model
from docx import Document

# 1. Download XOM Data (Jan 1, 2000 to Dec 31, 2010)
xom = yf.download("XOM", start="2000-01-01", end="2010-12-31")

# Flatten columns if yfinance returns a MultiIndex
if isinstance(xom.columns, pd.MultiIndex):
    xom.columns = xom.columns.get_level_values(0)

# 2. Calculate Daily Percentage Returns (Log-Difference)
# FIXED: We now use 'Close' because yfinance automatically adjusted it!
xom['Log_Return'] = 100 * (np.log(xom['Close']) - np.log(xom['Close'].shift(1)))

# Drop the first NaN row
returns = xom['Log_Return'].dropna()

print("Data downloaded and returns calculated successfully!")

/tmp/ipykernel_1413/425181484.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  xom = yf.download("XOM", start="2000-01-01", end="2010-12-31")
[*********************100%***********************]  1 of 1 completed

Data downloaded and returns calculated successfully!


In [5]:
# Demean the returns (mean equation)
mean_return = returns.mean()
resid = returns - mean_return
resid_sq = resid**2

# Engle's LM Test for ARCH effects (testing 5 lags)
# Returns: LM statistic, p-value, F statistic, F p-value
lm_stat, lm_pval, f_stat, f_pval = het_arch(resid, nlags=5)
print(f"Engle's LM Test p-value: {lm_pval:.4f}")

# Ljung-Box Q^2 test on squared residuals (testing 5 lags)
lb_test = acorr_ljungbox(resid_sq, lags=[5], return_df=True)
lb_pval = lb_test['lb_pvalue'].iloc[0]
print(f"Ljung-Box Q^2 Test p-value: {lb_pval:.4f}")

if lm_pval < 0.05 and lb_pval < 0.05:
    print("Conclusion: Significant ARCH effects present. Proceed with GARCH.")
else:
    print("Conclusion: No significant ARCH effects detected.")

Engle's LM Test p-value: 0.0000
Ljung-Box Q^2 Test p-value: 0.0000
Conclusion: Significant ARCH effects present. Proceed with GARCH.


In [6]:
# Define and fit the GARCH(1,1) model
# mean='Constant' implies we are estimating a constant mean alongside the variance
garch_mod = arch_model(returns, vol='Garch', p=1, q=1, mean='Constant')
garch_res = garch_mod.fit(disp='off')

print(garch_res.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:             Log_Return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -4970.01
Distribution:                  Normal   AIC:                           9948.02
Method:            Maximum Likelihood   BIC:                           9971.72
                                        No. Observations:                 2765
Date:                Thu, Mar 05 2026   Df Residuals:                     2764
Time:                        19:57:03   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0659  2.531e-02      2.605  9.191e-03 [1.632e-0

In [7]:
# Get standardized residuals (residuals divided by conditional volatility)
std_resid = garch_res.resid / garch_res.conditional_volatility
std_resid_sq = std_resid**2

# Ljung-Box Q^2 test on squared STANDARDIZED residuals
lb_post_test = acorr_ljungbox(std_resid_sq.dropna(), lags=[5], return_df=True)
lb_post_pval = lb_post_test['lb_pvalue'].iloc[0]

print(f"Post-Estimation Ljung-Box Q^2 p-value: {lb_post_pval:.4f}")
if lb_post_pval > 0.05:
    print("Conclusion: Model is well-specified. No remaining ARCH effects.")
else:
    print("Conclusion: Model may need more lags. ARCH effects remain.")

Post-Estimation Ljung-Box Q^2 p-value: 0.9093
Conclusion: Model is well-specified. No remaining ARCH effects.


In [8]:
# Initialize Word Document
doc = Document()
doc.add_heading('Exxon-Mobil (XOM) Daily Returns Analysis (2000-2010)', 0)

# Section 1: Pre-Estimation
doc.add_heading('1. Testing for ARCH Effects', level=1)
doc.add_paragraph(
    f"Prior to modeling, we tested the residuals of the mean equation for volatility clustering. "
    f"Engle's LM test yielded a p-value of {lm_pval:.4f}, and the Ljung-Box Q-squared test on "
    f"the squared residuals yielded a p-value of {lb_pval:.4f}. Because both p-values are less than 0.05, "
    f"we reject the null hypothesis of no ARCH effects. This confirms that XOM daily returns exhibit "
    f"significant volatility clustering and require an ARCH/GARCH framework."
)

# Section 2: Model Estimation
doc.add_heading('2. Model Choice and Estimation: GARCH(1,1)', level=1)
doc.add_paragraph(
    "Because ARCH effects are present, we estimated a GARCH(1,1) model. "
    "A GARCH(1,1) model solves the issue of needing infinite ARCH lags by capturing both "
    "short-term volatility spikes (the ARCH term) and long-term persistence (the GARCH term) "
    "efficiently with only two parameters."
)

# Add the GARCH summary table as text (to keep it simple in Word)
doc.add_paragraph("GARCH(1,1) Estimation Results:")
doc.add_paragraph(garch_res.summary().as_text())

# Section 3: Post-Estimation
doc.add_heading('3. Post-Estimation Specification Test', level=1)
doc.add_paragraph(
    f"To defend the GARCH(1,1) specification, we generated the squared standardized residuals "
    f"from the model and ran the Ljung-Box Q-squared test. The resulting p-value is {lb_post_pval:.4f}. "
    f"Because this is greater than 0.05, we fail to reject the null hypothesis, concluding that "
    f"the GARCH(1,1) model successfully captured the ARCH effects in the XOM returns, leaving "
    f"no remaining conditional heteroskedasticity."
)

# Save the document
filename = 'XOM_Volatility_Analysis_Python.docx'
doc.save(filename)
print(f"Success! Interpretations and results saved to {filename}")

Success! Interpretations and results saved to XOM_Volatility_Analysis_Python.docx
